In [ ]:
print("WEX427 Project Started")

In [ ]:
import pandas as pd

df = pd.read_csv("customer_support_tickets_200k.csv")

print("Satır sayısı:", len(df))
print("Sütun sayısı:", len(df.columns))

df.head()

In [ ]:
def create_label(category):
    if category == "Feature Request":
        return "Sales Inquiry"
    else:
        return "Complaint"

df["label"] = df["category"].apply(create_label)

print(df["label"].value_counts())

In [ ]:
df[["category","label"]].head(20)

In [ ]:
df_model = df[["issue_description", "label"]].copy()

print(df_model.head())
print(df_model["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model["issue_description"].values
y = df_model["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))
print("Train labels:")
print(pd.Series(y_train).value_counts())
print("Test labels:")
print(pd.Series(y_test).value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)
print(X_test_vec.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

predictions = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")

model.fit(X_train_vec, y_train)

predictions = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, predictions)

plt.show()

In [ ]:
def route_message(message):

    text_vec = vectorizer.transform([message])

    prediction = model.predict(text_vec)[0]

    if prediction == "Sales Inquiry":
        return "AI Sales Assistant"

    elif prediction == "Complaint":
        return "Customer Support System"

    else:
        return "Spam Detection System"

In [ ]:
print(route_message("Can you tell me the price of your premium plan?"))
print(route_message("I want a refund for my last payment"))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_tensor = torch.tensor(X_train_vec.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_vec.toarray(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)
print(label_encoder.classes_)

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(TextClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.network(x)


input_size = X_train_tensor.shape[1]
hidden_size = 64
num_classes = len(label_encoder.classes_)

pytorch_model = TextClassifier(input_size, hidden_size, num_classes)

print(pytorch_model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(pytorch_model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = pytorch_model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    loss.backward()

    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

pytorch_model.eval()

with torch.no_grad():
    outputs = pytorch_model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)

accuracy = accuracy_score(
    y_test_tensor.numpy(),
    predicted.numpy()
)

print("PyTorch Accuracy:", accuracy)

print(
    classification_report(
        y_test_tensor.numpy(),
        predicted.numpy(),
        target_names=label_encoder.classes_
    )
)

In [ ]:
# Modeli kaydet
torch.save(pytorch_model.state_dict(), "pytorch_text_classifier.pth")

# Temizlenmiş veriyi kaydet
df_model.to_csv("processed_customer_tickets.csv", index=False)

print("Files saved successfully.")

In [ ]:
spam_texts = [
    "Congratulations! You won a free prize, click here to claim it.",
    "You have won a lottery reward, send your bank details.",
    "Earn money fast from home with no experience.",
    "Free crypto giveaway, click this link now.",
    "Your account won a bonus gift card.",
    "Limited offer! Claim your free iPhone today.",
    "Click here to receive your cash reward.",
    "You are selected for a free vacation prize.",
    "Win a new laptop by clicking this link.",
    "Urgent! Send your password to verify your account."
]

spam_rows = []

for i in range(1000):
    spam_rows.append({
        "issue_description": spam_texts[i % len(spam_texts)],
        "label": "Spam"
    })

spam_df = pd.DataFrame(spam_rows)

df_model_3class = pd.concat([df_model, spam_df], ignore_index=True)

print(df_model_3class["label"].value_counts())

In [ ]:
X = df_model_3class["issue_description"].values
y = df_model_3class["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(pd.Series(y_train).value_counts())
print(pd.Series(y_test).value_counts())

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)
print(X_test_vec.shape)

In [ ]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_tensor = torch.tensor(X_train_vec.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_vec.toarray(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

print(label_encoder.classes_)
print(X_train_tensor.shape)
print(y_train_tensor.shape)

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.network(x)

input_size = X_train_tensor.shape[1]
hidden_size = 64
num_classes = len(label_encoder.classes_)

pytorch_model = TextClassifier(
    input_size,
    hidden_size,
    num_classes
)

print(pytorch_model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    pytorch_model.parameters(),
    lr=0.001
)

epochs = 10

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = pytorch_model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {loss.item():.4f}"
    )

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

pytorch_model.eval()

with torch.no_grad():

    outputs = pytorch_model(X_test_tensor)

    _, predicted = torch.max(outputs, 1)

accuracy = accuracy_score(
    y_test_tensor.numpy(),
    predicted.numpy()
)

print("Accuracy:", accuracy)

print(
    classification_report(
        y_test_tensor.numpy(),
        predicted.numpy(),
        target_names=label_encoder.classes_
    )
)